# ПУНКТ 3. Разбиение на обучающую и тестовую выборки 

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_pickle('../data/clean_dataset.pkl')
print("Размер датасета:", df.shape)
print(df['price_range'].value_counts())

X = df.drop('price_range', axis=1)
y = df['price_range']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=67,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Размер датасета: (2000, 22)
price_range
1    500
2    500
3    500
0    500
Name: count, dtype: int64
Train: (1500, 21) Test: (500, 21)


# ПУНКТ 4. Списки числовых и категориальных признаков

In [2]:
cat_features = ['blue', 'dual_sim', 'four_g', 'three_g', 'touch_screen', 'wifi']

num_features = [col for col in X_train.columns if col not in cat_features]

print("Числовых признаков:", len(num_features))
print(num_features)
print("Категориальных признаков:", len(cat_features))
print(cat_features)

Числовых признаков: 15
['battery_power', 'clock_speed', 'fc', 'int_memory', 'm_dep', 'mobile_wt', 'n_cores', 'pc', 'px_height', 'px_width', 'ram', 'sc_h', 'sc_w', 'talk_time', 'screen_area']
Категориальных признаков: 6
['blue', 'dual_sim', 'four_g', 'three_g', 'touch_screen', 'wifi']


# ПУНКТ 5. Pipeline обработки признаков и обучения модели

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.ensemble import RandomForestClassifier

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', TargetEncoder(random_state=67), cat_features),
    ],
    remainder='drop'
)

baseline = Pipeline(steps=[
    ('transform', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=67, n_jobs=-1)),
])

baseline.fit(X_train, y_train)

,steps,"[('transform', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# ПУНКТ 6. Baseline-модель

In [4]:
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score)

def calc_metrics(y_true, y_pred, y_proba):
    return {
        'precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_proba, multi_class='ovr', average='weighted'),
    }

y_pred = baseline.predict(X_test)
y_proba = baseline.predict_proba(X_test)

baseline_metrics = calc_metrics(y_test, y_pred, y_proba)
print(baseline_metrics)

{'precision': 0.8627417721895569, 'recall': 0.864, 'f1': 0.863179629185696, 'roc_auc': 0.9684879999999999}


# ПУНКТ 9. Логирование baseline в MlFLow

In [5]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# Подключаемся к запущенному локальному серверу MLflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_registry_uri("http://127.0.0.1:5000")

EXPERIMENT_NAME = "mobile_price_experiment"
MODEL_NAME = "mobile_price_model"

## Создаём эксперимент

In [6]:
experiment_id = mlflow.set_experiment(EXPERIMENT_NAME).experiment_id
print("experiment_id:", experiment_id)

2026/10/27 02:29:12 INFO mlflow.tracking.fluent: Experiment with name 'mobile_price_experiment' does not exist. Creating a new experiment.


experiment_id: 1


## Готовим сигнатуру и пример входа

In [7]:
signature = infer_signature(X_train.head(5))
input_example = X_train.head(5)

req_file = '../requirements.txt'

/home/mainuser/mainuser/chik_proj/.venv_chik_proj/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


## Логируем baseline

In [8]:
with mlflow.start_run(run_name='baseline', experiment_id=experiment_id) as run:
    mlflow.sklearn.log_model(
        baseline,  
        artifact_path="models",
        signature=signature,
        input_example=input_example,
        pip_requirements=req_file,
        registered_model_name=MODEL_NAME,
    )
    mlflow.log_metrics(baseline_metrics)
    mlflow.log_params({'model': 'RandomForest', 'n_estimators': 100})
    print("baseline run_id:", run.info.run_id)

Successfully registered model 'mobile_price_model'.
2026/10/27 02:29:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: mobile_price_model, version 1
Created version '1' of model 'mobile_price_model'.
2026/10/27 02:29:20 INFO mlflow.tracking._tracking_service.client: 🏃 View run baseline at: http://127.0.0.1:5000/#/experiments/1/runs/6e6d0ca8b7b14cfea48f795462374561.
2026/10/27 02:29:20 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


baseline run_id: 6e6d0ca8b7b14cfea48f795462374561


# ПУНКТ 10. Генерация новых признаков

In [9]:
from sklearn.preprocessing import PolynomialFeatures, KBinsDiscretizer

poly_cols = ['battery_power', 'ram', 'px_width']
bin_cols  = ['int_memory', 'mobile_wt']

poly_pipe = Pipeline(steps=[
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scale', StandardScaler()),
])

preprocessor_fe = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', TargetEncoder(random_state=67), cat_features),
        ('poly', poly_pipe, poly_cols),
        ('bins', KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile'), bin_cols),
    ],
    remainder='drop'
)

X_train_fe_sklearn = X_train.copy()

X_train_fe_sklearn = pd.DataFrame(
    preprocessor_fe.fit_transform(X_train, y_train),
    columns=preprocessor_fe.get_feature_names_out(),
)
print("После FE:", X_train_fe_sklearn.shape)

pipeline_fe = Pipeline(steps=[
    ('transform', preprocessor_fe),
    ('model', RandomForestClassifier(n_estimators=100, random_state=67, n_jobs=-1)),
])
pipeline_fe.fit(X_train, y_train)

y_pred = pipeline_fe.predict(X_test)
y_proba = pipeline_fe.predict_proba(X_test)
fe_metrics = calc_metrics(y_test, y_pred, y_proba)
print("FE metrics:", fe_metrics)

/home/mainuser/mainuser/chik_proj/.venv_chik_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/home/mainuser/mainuser/chik_proj/.venv_chik_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


После FE: (1500, 50)
FE metrics: {'precision': 0.9090599798387098, 'recall': 0.908, 'f1': 0.9083677777982949, 'roc_auc': 0.9889946666666666}


## Сохраняем список столбцов

In [10]:
fe_columns = list(preprocessor_fe.get_feature_names_out())

with open('fe_columns.txt', 'w') as f:
    f.write('\n'.join(fe_columns))

print("Всего признаков после FE:", len(fe_columns))

Всего признаков после FE: 50


## Логируем FE

In [11]:
with mlflow.start_run(run_name='fe_sklearn', experiment_id=experiment_id) as run:
    mlflow.sklearn.log_model(pipeline_fe, artifact_path="models",signature=signature, 
                             input_example=input_example, pip_requirements=req_file,
    )
    mlflow.log_metrics(fe_metrics)
    mlflow.log_artifact('fe_columns.txt')
    mlflow.log_params({'poly_cols': str(poly_cols), 'bin_cols': str(bin_cols)})

2026/10/27 02:29:36 INFO mlflow.tracking._tracking_service.client: 🏃 View run fe_sklearn at: http://127.0.0.1:5000/#/experiments/1/runs/38e7b2b257a4477c8552a00e432d8592.
2026/10/27 02:29:36 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


# ПУНКТ 12. Отбор N важных признаков

## 12.1 Готовим трансформированный датафрейм

In [12]:
from sklearn.feature_selection import RFE

X_train_fe = X_train_fe_sklearn.copy()

X_test_fe = pd.DataFrame(
    preprocessor_fe.transform(X_test),
    columns=preprocessor_fe.get_feature_names_out(),
)

print("Признаков после FE:", X_train_fe.shape[1])

Признаков после FE: 50


## 12.2 Отбираем ~50% признаков

In [13]:
N_FEATURES = int(0.5 * X_train_fe.shape[1])
print("Оставляем:", N_FEATURES, "признаков из", X_train_fe.shape[1])

rfe = RFE(
    estimator=RandomForestClassifier(n_estimators=100, random_state=67, n_jobs=-1),
    n_features_to_select=N_FEATURES,
    step=0.1,
)
rfe.fit(X_train_fe, y_train)

rfe_columns = X_train_fe.columns[rfe.support_].tolist()
rfe_idx  = list(rfe.support_)

with open('rfe_columns.txt', 'w') as f:
    f.write('\n'.join(rfe_columns))
with open('rfe_idx.txt', 'w') as f:
    f.write(str(rfe_idx))

print("Отобрано:", rfe_columns)

Оставляем: 25 признаков из 50
Отобрано: ['num__battery_power', 'num__fc', 'num__int_memory', 'num__m_dep', 'num__mobile_wt', 'num__n_cores', 'num__pc', 'num__px_height', 'num__px_width', 'num__ram', 'num__sc_h', 'num__sc_w', 'num__talk_time', 'num__screen_area', 'cat__blue_2', 'cat__wifi_1', 'poly__battery_power', 'poly__ram', 'poly__px_width', 'poly__battery_power^2', 'poly__battery_power ram', 'poly__battery_power px_width', 'poly__ram^2', 'poly__ram px_width', 'poly__px_width^2']


## 12.3 Обучаем модель на отобранных признаках

In [14]:
from sklearn.preprocessing import FunctionTransformer

def select_columns(X):
    return X[:, rfe.support_]

selector = FunctionTransformer(select_columns)

pipeline_rfe = Pipeline(steps=[
    ('transform', preprocessor_fe),
    ('selector', selector),
    ('model', RandomForestClassifier(n_estimators=100, random_state=67, n_jobs=-1)),
])

pipeline_rfe.fit(X_train, y_train)

y_pred  = pipeline_rfe.predict(X_test)
y_proba = pipeline_rfe.predict_proba(X_test)
rfe_metrics = calc_metrics(y_test, y_pred, y_proba)
print("RFE metrics:", rfe_metrics)

/home/mainuser/mainuser/chik_proj/.venv_chik_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


RFE metrics: {'precision': 0.9139373526027599, 'recall': 0.912, 'f1': 0.9125043957707948, 'roc_auc': 0.9897706666666668}


## 12.4 Логируем

In [15]:
with mlflow.start_run(run_name='rfe_sklearn', experiment_id=experiment_id) as run:
    mlflow.sklearn.log_model(pipeline_rfe, artifact_path="models",signature=signature, input_example=input_example, 
                             pip_requirements=req_file,
    )
    mlflow.log_metrics(rfe_metrics)
    mlflow.log_artifact('rfe_columns.txt')
    mlflow.log_artifact('rfe_idx.txt')

2026/10/27 02:30:50 INFO mlflow.tracking._tracking_service.client: 🏃 View run rfe_sklearn at: http://127.0.0.1:5000/#/experiments/1/runs/18fcbd4d7ec943bcb2265848e402204c.
2026/10/27 02:30:50 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


# ПУНКТ 14. Лучшая модель

In [16]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, make_scorer

f1_scorer = make_scorer(f1_score, average='weighted', greater_is_better=True)

param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [10, 20, None],
    'model__max_features': [0.3, 0.5, 1.0],
}

gs = GridSearchCV(Pipeline(steps=[
        ('transform', preprocessor_fe),
        ('selector', selector),
        ('model', RandomForestClassifier(random_state=67)),
    ]),
    param_grid,
    cv=3,
    scoring=f1_scorer,
    n_jobs=-1,
    verbose=1,
)
gs.fit(X_train, y_train)

print("Лучшие параметры:", gs.best_params_)
print("Лучший f1 (cv):", gs.best_score_)

Fitting 3 folds for each of 27 candidates, totalling 81 fits


/home/mainuser/mainuser/chik_proj/.venv_chik_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/home/mainuser/mainuser/chik_proj/.venv_chik_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/home/mainuser/mainuser/chik_proj/.venv_chik_proj/lib/python3.10/site-packages/sklearn/preprocessi

Лучшие параметры: {'model__max_depth': 20, 'model__max_features': 0.3, 'model__n_estimators': 300}
Лучший f1 (cv): 0.9041398258851624


## Проверка на тесте

In [17]:
pipeline_gs = gs.best_estimator_
y_pred = pipeline_gs.predict(X_test)
y_proba = pipeline_gs.predict_proba(X_test)
gs_metrics = calc_metrics(y_test, y_pred, y_proba)
print("GridSearch test metrics:", gs_metrics)

GridSearch test metrics: {'precision': 0.9128874041954728, 'recall': 0.912, 'f1': 0.912247639653816, 'roc_auc': 0.9902613333333333}


## Логируем как Version 2

In [18]:
with mlflow.start_run(run_name='gridsearch', experiment_id=experiment_id) as run:
    mlflow.sklearn.log_model(
        pipeline_gs, artifact_path="models",
        signature=signature, input_example=input_example, pip_requirements=req_file,
        registered_model_name=MODEL_NAME,
    )
    mlflow.log_metrics(gs_metrics)
    mlflow.log_params(gs.best_params_)
    mlflow.log_artifact('rfe_columns.txt')

Registered model 'mobile_price_model' already exists. Creating a new version of this model...
2026/10/27 02:31:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: mobile_price_model, version 2
Created version '2' of model 'mobile_price_model'.
2026/10/27 02:31:58 INFO mlflow.tracking._tracking_service.client: 🏃 View run gridsearch at: http://127.0.0.1:5000/#/experiments/1/runs/61e0665756914c5799fa51c45ea1c978.
2026/10/27 02:31:58 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


# Пункт 16. Финальная модель на всей выборке + Production

## 16.1 Сравниваем все прогоны и берём лучший

In [19]:
all_results = {
    'baseline': baseline_metrics,
    'fe_sklearn': fe_metrics,
    'rfe_sklearn': rfe_metrics,
    'gridsearch': gs_metrics,
}

best_name = max(all_results, key=lambda k: all_results[k]['f1'])
print("Лучшая модель:", best_name, all_results[best_name])

Лучшая модель: rfe_sklearn {'precision': 0.9139373526027599, 'recall': 0.912, 'f1': 0.9125043957707948, 'roc_auc': 0.9897706666666668}


## 16.2 Обучаем лучший пайплайн на всей выборке

In [20]:
X_full = df.drop('price_range', axis=1)
y_full = df['price_range']

pipeline_prod = Pipeline(steps=[
    ('transform', preprocessor_fe),
    ('selector', selector),
    ('model', RandomForestClassifier(n_estimators=100, random_state=67, n_jobs=-1)),
])

pipeline_prod.fit(X_full, y_full)
print("Production-модель обучена на", X_full.shape[0], "объектах")

/home/mainuser/mainuser/chik_proj/.venv_chik_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


Production-модель обучена на 2000 объектах


## 16.3 Логируем + регистрируем + ставим Production

In [21]:
from mlflow.tracking import MlflowClient

with mlflow.start_run(run_name='production', experiment_id=experiment_id) as run:
    mlflow.sklearn.log_model(
        pipeline_prod, artifact_path="models",
        signature=signature, input_example=input_example, pip_requirements=req_file,
        registered_model_name=MODEL_NAME,
    )
    mlflow.log_artifact('rfe_columns.txt')
    prod_run_id = run.info.run_id
    print("Production run_id:", prod_run_id)

client = MlflowClient()
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest_version = max(int(v.version) for v in versions)

client.transition_model_version_stage(
    name=MODEL_NAME,
    version=latest_version,
    stage="Production",
)
print(f"Version {latest_version} Production")

Registered model 'mobile_price_model' already exists. Creating a new version of this model...
2026/10/27 02:33:23 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: mobile_price_model, version 3
Created version '3' of model 'mobile_price_model'.
2026/10/27 02:33:23 INFO mlflow.tracking._tracking_service.client: 🏃 View run production at: http://127.0.0.1:5000/#/experiments/1/runs/e7abb0fb22744d8c9724d5716e591657.
2026/10/27 02:33:23 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.
/tmp/ipykernel_56873/2292236398.py:17: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages


Production run_id: e7abb0fb22744d8c9724d5716e591657
Version 3 Production
